In [1]:
# test ritardi
import sys
import os
import numpy as np
from scipy import stats

# Navigate from test/ to src/ using relative path
notebook_dir = os.path.dirname(os.path.abspath('.'))
src_path = os.path.join(notebook_dir,  'src')
sys.path.insert(0, src_path)

from common_lib import CommonLib
from stats_lib import DelaysStats

######################
def count_values_with_coverage(delays_summary, lower_bounds, upper_bounds):
    """
    Calculate coverage ratio within value ranges for each column.
    
    Uses vectorized NumPy operations for performance.
    
    Args:
        delays_summary: 2D array with 4 columns
                        Col 0: min delays [U(k)]
                        Col 1: max delays [V(k)]
                        Col 2: mean delays [W(k)]
                        Col 3: tot delays [Y(k)]
        lower_bounds: Array of 4 lower bounds
        upper_bounds: Array of 4 upper bounds
    
    Returns:
        Array of 4 coverage ratios (0.0 to 1.0)
    """
    result = np.zeros(4, dtype=np.float64)
    
    try:
        delays_summary = np.asarray(delays_summary, dtype=np.int64)
        lower_bounds = np.asarray(lower_bounds, dtype=np.float64)
        upper_bounds = np.asarray(upper_bounds, dtype=np.float64)
        
        if delays_summary.ndim != 2 or delays_summary.shape[1] != 4:
            return result
        if lower_bounds.shape != (4,) or upper_bounds.shape != (4,):
            return result
        
        if delays_summary.size == 0 or delays_summary.shape[0] == 0:
            return result
        
        if np.any(lower_bounds > upper_bounds):
            return result
        
        for col_idx in range(4):
            column_data = delays_summary[:, col_idx]
            
            unique, counts = np.unique(column_data, return_counts=True)
            
            max_value = int(unique.max()) if unique.size > 0 else 0
            count_array = np.zeros(max_value + 1, dtype=np.int64)
            
            count_array[unique] = counts
            
            min_idx = np.clip(np.round(lower_bounds[col_idx]).astype(np.int64), 0, len(count_array) - 1)
            max_idx = np.clip(np.round(upper_bounds[col_idx]).astype(np.int64), 0, len(count_array) - 1)
            
            if min_idx > max_idx:
                result[col_idx] = 0.0
                continue
            
            # Calculate the sum of counts within the specified range
            sum_range = np.sum(count_array[min_idx:max_idx + 1])
            # Calculate the total sum of counts for normalization
            total_sum = np.sum(count_array)
            
            # Calculate coverage ratio, handling division by zero
            result[col_idx] = sum_range / total_sum if total_sum > 0 else 0.0
        
        return result
        
    except (TypeError, ValueError, IndexError, OverflowError):
        return result

######################

data_file_name = "../data/estrazioni_2026_2006_copy.csv"
clib = CommonLib(data_file_name)
dlib = DelaysStats()

# Popolazione
history_data = clib.load_data() # Carica le estrazioni da un file CSV

# td = []
# hyp_max_delay = 200 # Valore ipotetico di ritardo massimo per la simulazione
# for i in range(len(history_data) - hyp_max_delay):  
#     #print(f"Estrazione {i+1}: {history_data[i]}")
#     base = history_data[i]  # Prendi l'ultima estrazione come base per il calcolo dei ritardi
#     sample_data = history_data[i+1:i+hyp_max_delay]  # Prendi le prime 100 estrazioni per il campione   
#     delays = dlib.base_delays(base, sample_data)

#     #print(f"Ritardi: {np.sort(delays)}")
#     min_delay = np.min(delays)
#     max_delay = np.max(delays)
#     mean_delay = np.round(np.mean(delays), 2)
#     tot_delay = np.sum(delays)
#     d = np.array([min_delay, max_delay, mean_delay, tot_delay])
#     #print(f"Ritardo minimo: {min_delay} - massimo: {max_delay} - medio: {mean_delay}")
#     td.append(d)
    
# mind = np.round(np.min(td, axis=0), 0)
# maxd = np.round(np.max(td, axis=0), 0)
# print(f"Ritardo totale \n- minimo: {mind} \n- massimo: {maxd}")

# u = np.round(np.mean(td, axis=0), 0)
# x = np.round(np.std(td, axis=0), 0)
# sk = np.round(stats.skew(td, axis=0), 2)

# q1 = np.round(np.quantile(td, 0.25, axis=0), 0)
# q3 = np.round(np.quantile(td, 0.75, axis=0), 0)

# # TODO: aggiungere calcolo copertura

# print(f"Ritardo totale \n- media: {u} \n- deviazione standard: {x} \n- skewness: {sk} \n- q1: {q1} \n- q3: {q3}")

# min_bounds_simm = np.round(u - x, 2)
# max_bounds_simm = np.round(u + x, 2)
# coverages_simm = np.round((count_values_with_coverage(td, min_bounds_simm, max_bounds_simm)) * 100, 2)
# print(f"Ritardo totale - copertura intervallo media ± deviazione standard: {coverages_simm}%")

# min_bounds_asimm = q1
# max_bounds_asimm = q3
# coverages_asimm = np.round((count_values_with_coverage(td, min_bounds_asimm, max_bounds_asimm)) * 100, 2)
# print(f"Ritardo totale - copertura intervallo IQR: {coverages_asimm}%")

# results = {}
# results['description'] = "Ritardi"
# measures = {}
# measures['values'] = [{}] * 4
# for j in range(0, 4):
#     measures['values'][j] = {
#         'min': mind[j],
#         'max': maxd[j],
#         'mean': u[j],
#         'std_dev': x[j],
#         'skewness': sk[j],
#         'q1': q1[j],
#         'q3': q3[j],
#         'simmetrica': {
#             'min': min_bounds_simm[j],
#             'max': max_bounds_simm[j],
#             'coverage': coverages_simm[j]
#         },
#         'asimmetrica': {
#             'min': min_bounds_asimm[j],
#             'max': max_bounds_asimm[j],
#             'coverage': coverages_asimm[j]
#         }
#     }
# results['measures'] = measures
results = dlib.get_delays_dict(history_data)
print(results)

Estrazione 1: [ 8 13 25 60 72 74]
Ritardi: [ 2  3  3  8 13 47]
Ritardo minimo: 2 - massimo: 47 - medio: 12.67
Estrazione 2: [ 3  7 41 56 65 83]
Ritardi: [ 2 11 12 17 22 38]
Ritardo minimo: 2 - massimo: 38 - medio: 17.0
Estrazione 3: [14 21 24 52 80 86]
Ritardi: [ 3  6 13 15 19 20]
Ritardo minimo: 3 - massimo: 20 - medio: 12.67
Estrazione 4: [44 49 60 69 73 85]
Ritardi: [ 1  7  8 12 19 55]
Ritardo minimo: 1 - massimo: 55 - medio: 17.0
Estrazione 5: [20 29 56 68 72 74]
Ritardi: [2 2 4 6 7 9]
Ritardo minimo: 2 - massimo: 9 - medio: 5.0
Estrazione 6: [ 1 11 40 73 75 81]
Ritardi: [ 0 12 12 13 19 26]
Ritardo minimo: 0 - massimo: 26 - medio: 13.67
Estrazione 7: [11 19 24 66 82 88]
Ritardi: [ 1  2  7  8 18 47]
Ritardo minimo: 1 - massimo: 47 - medio: 13.83
Estrazione 8: [31 33 61 68 71 72]
Ritardi: [ 2  3  5 15 16 18]
Ritardo minimo: 2 - massimo: 18 - medio: 9.83
Estrazione 9: [35 42 45 53 55 88]
Ritardi: [ 4  8  9 11 25 28]
Ritardo minimo: 4 - massimo: 28 - medio: 14.17
Estrazione 10: [11 13 